# ⚽ Entrena tu equipo de fútbol 2 vs 2 (VMAS + BenchMARL)

Vas a entrenar **2 agentes** (tu equipo azul) con **IQL (Independent Q-Learning, basado en DQN)** contra un rival controlado por la IA heurística de VMAS.
Al final se genera un archivo `equipo_<nombre>.pt` que debes **enviar al profesor**: ese archivo se usará en el cuaderno de inferencia para enfrentar tu equipo contra el de otro estudiante (el primero en llegar a 5 goles gana).

**Pasos:** `Entorno > Cambiar tipo de entorno de ejecución > GPU` → ejecuta las celdas en orden.

> ⚠️ Solo puedes modificar la celda **"Parámetros que SÍ puedes cambiar"**. La arquitectura de la red y el entorno están fijos para que todos los equipos sean compatibles.

In [ ]:
!pip install -q vmas==1.5.2 benchmarl==1.5.2

## 1. Identifica a tu equipo

In [ ]:
NOMBRE_EQUIPO = "Los_Cracks"      # sin espacios ni tildes
INTEGRANTES   = "Nombre Apellido"  # quién entrena este equipo
SEMILLA       = 0

## 2. Parámetros que SÍ puedes cambiar
Experimenta con ellos para mejorar a tu equipo.

In [ ]:
MAX_FRAMES     = 1_200_000   # más frames = más entrenamiento (cada 6000 frames = 1 iteración). Con GPU, 1.2M puede tardar del orden de decenas de minutos.
LR             = 1e-4        # tasa de aprendizaje
GAMMA          = 0.99        # factor de descuento
EPS_INICIAL    = 0.8         # exploración inicial (epsilon-greedy)
EPS_FINAL      = 0.01        # exploración final
DIFICULTAD_IA  = 0.7         # 0..1, velocidad del rival de entrenamiento (1.0 = el más difícil)

# Recompensas (reward shaping)
RECOMPENSA_GOL            = 100.0  # recompensa al marcar
SHAPING_BALON_A_PORTERIA  = 10.0   # premio por acercar el balón a la portería rival
SHAPING_JUGADOR_A_BALON   = 0.1    # premio por acercarse al balón

## 3. Configuración FIJA (no tocar)

In [ ]:
import torch, vmas
from benchmarl.algorithms import IqlConfig
from benchmarl.environments import VmasTask
from benchmarl.experiment import Experiment, ExperimentConfig
from benchmarl.models.mlp import MlpConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

# ---- FIJO: debe ser idéntico para todos los equipos ----
TASK_FIJO = dict(
    max_steps=500,
    n_blue_agents=2, n_red_agents=2,
    ai_red_agents=True, ai_blue_agents=False,
    enable_shooting=False,            # acciones: solo movimiento (9 acciones discretas)
    observe_teammates=True, observe_adversaries=True, dict_obs=False,
)
TASK_ENTRENAMIENTO = dict(
    TASK_FIJO,
    ai_strength=DIFICULTAD_IA,
    scoring_reward=RECOMPENSA_GOL,
    pos_shaping_factor_ball_goal=SHAPING_BALON_A_PORTERIA,
    pos_shaping_factor_agent_ball=SHAPING_JUGADOR_A_BALON,
)

task = VmasTask.FOOTBALL.get_from_yaml()
task.config.update(TASK_ENTRENAMIENTO)

algorithm_config = IqlConfig.get_from_yaml()     # IQL = DQN independiente por agente
model_config     = MlpConfig.get_from_yaml()     # MLP 2x256 (fijo)

cfg = ExperimentConfig.get_from_yaml()
cfg.sampling_device = DEVICE
cfg.train_device    = DEVICE
cfg.buffer_device   = DEVICE
cfg.share_policy_params = True                   # fijo
cfg.prefer_continuous_actions = False            # IQL/DQN => acciones discretas
cfg.max_n_frames = MAX_FRAMES
cfg.max_n_iters  = None
cfg.lr = LR
cfg.gamma = GAMMA
cfg.exploration_eps_init = EPS_INICIAL
cfg.exploration_eps_end  = EPS_FINAL
cfg.off_policy_memory_size = 500_000
cfg.evaluation = True
cfg.evaluation_interval = 120_000
cfg.evaluation_episodes = 5
cfg.render = False
cfg.loggers = ["csv"]
cfg.create_json = False
cfg.checkpoint_at_end = False
cfg.save_folder = "/content/entrenamiento"

import os; os.makedirs(cfg.save_folder, exist_ok=True)

experiment = Experiment(
    task=task, algorithm_config=algorithm_config, model_config=model_config,
    seed=SEMILLA, config=cfg,
)
GRUPO = list(experiment.group_map.keys())[0]
print("Grupo de agentes:", GRUPO, experiment.group_map)

## 4. ¡Entrenar! 🏋️
Verás una barra de progreso con el retorno medio. Cuanto más alto, mejor.

In [ ]:
experiment.run()

## 5. Prueba rápida: tu equipo vs la IA
Se juegan 10 episodios de hasta 500 pasos y se cuentan los goles.

In [ ]:
from tensordict import TensorDict
from torchrl.envs.utils import set_exploration_type, ExplorationType

def elegir_acciones(policy, obs_list, grupo=GRUPO):
    obs = torch.stack(obs_list, dim=1)                    # [envs, agentes, obs_dim]
    n, na = obs.shape[0], obs.shape[1]
    td = TensorDict({grupo: TensorDict({"observation": obs}, batch_size=[n, na])}, batch_size=[n])
    with torch.no_grad(), set_exploration_type(ExplorationType.DETERMINISTIC):
        td = policy(td)
    return td.get((grupo, "action")).reshape(n, na)

env_test = vmas.make_env(
    scenario="football", num_envs=1, device=DEVICE, continuous_actions=False,
    max_steps=TASK_FIJO["max_steps"], seed=123, **TASK_FIJO,
)
OBS_DIM = env_test.reset()[0].shape[-1]
L = env_test.scenario.pitch_length

def evaluar_vs_ia(episodios=10):
    a_favor = en_contra = sin_gol = 0
    for _ in range(episodios):
        obs = env_test.reset_at(0)
        while True:
            acc = elegir_acciones(experiment.policy, obs)
            obs, _, done, _ = env_test.step([acc[:, i:i+1] for i in range(2)])
            if done.any():
                x = env_test.scenario.ball.state.pos[0, 0].item()
                if x > L/2: a_favor += 1
                elif x < -L/2: en_contra += 1
                else: sin_gol += 1
                break
    print(f"Goles a favor: {a_favor} | en contra: {en_contra} | episodios sin gol: {sin_gol}  (de {episodios})")

evaluar_vs_ia(10)

## 6. Guardar y descargar tu equipo 💾
Envía el archivo `.pt` descargado al profesor.

In [ ]:
import vmas as _v, benchmarl as _b, torchrl as _t
archivo = f"equipo_{NOMBRE_EQUIPO}.pt"
torch.save({
    "formato": "iql-football-2v2-v1",
    "equipo": NOMBRE_EQUIPO,
    "integrantes": INTEGRANTES,
    "grupo": GRUPO,
    "obs_dim": int(OBS_DIM),
    "n_acciones": 9,
    "frames_entrenados": int(experiment.total_frames),
    "versiones": {"vmas": _v.__version__, "torchrl": _t.__version__},
    "policy_state_dict": {k: v.detach().cpu() for k, v in experiment.policy.state_dict().items()},
}, archivo)
print("Guardado:", archivo)

from google.colab import files
files.download(archivo)